In [ ]:
!pip install transformers==4.40.2 peft==0.10.0 accelerate==0.30.1 trl==0.8.6

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.distributions import Categorical
from copy import deepcopy
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Load YOUR trained weights here
base_model.load_state_dict(torch.load("sft_model.pt"))

ref_model = deepcopy(base_model).to(device)
for p in ref_model.parameters():
    p.requires_grad = False

reward_model = torch.load("reward_model.pt")

In [ ]:
class PolicyWithValue(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.value_head = nn.Linear(base_model.config.n_embd, 1)

    def forward(self, input_ids):
        outputs = self.model.transformer(input_ids)
        hidden = outputs.last_hidden_state

        logits = self.model.lm_head(hidden)
        values = self.value_head(hidden).squeeze(-1)

        return logits, values

policy = PolicyWithValue(base_model).to(device)
ref_policy = PolicyWithValue(ref_model).to(device)

for p in ref_policy.parameters():
    p.requires_grad = False

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states = []
        self.actions = []
        self.logprobs = []
        self.values = []
        self.rewards = []
        self.masks = []
        self.returns = []

In [ ]:
def collect_trajectory(prompt, max_len=50):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    generated = input_ids.clone()

    actions, logprobs, values, masks = [], [], [], []

    for _ in range(max_len):

        logits, value = policy(generated)

        logits = logits[:, -1, :]
        value = value[:, -1]

        probs = torch.softmax(logits, dim=-1)
        dist = Categorical(probs)

        action = dist.sample()

        logprob = dist.log_prob(action)

        generated = torch.cat([generated, action.unsqueeze(0)], dim=1)

        actions.append(action)
        logprobs.append(logprob)
        values.append(value)
        masks.append(torch.tensor([1.0]).to(device))

    return generated, actions, logprobs, values, masks

In [ ]:
def compute_reward(prompt, text):
    inputs = tokenizer(prompt + " " + text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        reward = reward_model(**inputs)

    return reward.squeeze()

In [ ]:
def compute_gae(rewards, values, masks, gamma=0.99, lam=0.95):

    returns = []
    gae = 0
    next_value = 0

    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_value * masks[t] - values[t]
        gae = delta + gamma * lam * masks[t] * gae
        returns.insert(0, gae + values[t])
        next_value = values[t]

    return returns

In [ ]:
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-6)

def ppo_update(buffer, clip_eps=0.2, beta=0.01, epochs=4):

    states = torch.cat(buffer.states)
    actions = torch.cat(buffer.actions)
    old_logprobs = torch.cat(buffer.logprobs).detach()
    returns = torch.cat(buffer.returns).detach()
    values = torch.cat(buffer.values).detach()

    advantages = returns - values
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    for _ in range(epochs):

        logits, value_preds = policy(states)

        logits = logits[:, :-1, :]  # align with actions
        value_preds = value_preds[:, :-1]

        probs = torch.softmax(logits, dim=-1)
        dist = Categorical(probs)

        new_logprobs = dist.log_prob(actions)

        entropy = dist.entropy().mean()

        ratio = torch.exp(new_logprobs - old_logprobs)

        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages

        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = F.mse_loss(value_preds, returns)

        with torch.no_grad():
            ref_logits, _ = ref_policy(states)
            ref_probs = torch.softmax(ref_logits[:, :-1, :], dim=-1)

        kl = (probs * (torch.log(probs + 1e-8) - torch.log(ref_probs + 1e-8))).sum(-1).mean()

        loss = policy_loss + 0.5 * value_loss + beta * kl - 0.01 * entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
buffer = RolloutBuffer()

prompts = [
    "Explain AI simply",
    "What is RL?",
    "How does the internet work?"
]

for epoch in range(3):

    for prompt in prompts:

        gen_ids, actions, logprobs, values, masks = collect_trajectory(prompt)

        text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        reward = compute_reward(prompt, text)

        rewards = [reward for _ in actions]

        returns = compute_gae(rewards, values, masks)

        buffer.states.append(gen_ids)
        buffer.actions.append(torch.stack(actions))
        buffer.logprobs.append(torch.stack(logprobs))
        buffer.values.append(torch.stack(values))
        buffer.rewards.append(torch.tensor(rewards).to(device))
        buffer.masks.append(torch.stack(masks))
        buffer.returns.append(torch.stack(returns))

    ppo_update(buffer)
    buffer.clear()

    print(f"Epoch {epoch} done")

In [ ]:
def generate_responses(prompt, n=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = policy.model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        top_p=0.9,
        num_return_sequences=n
    )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [ ]:
def self_improve(prompts, top_k=2, random_k=1):

    new_data = []

    for prompt in prompts:

        responses = generate_responses(prompt, 6)

        scored = [(r, compute_reward(prompt, r).item()) for r in responses]
        scored.sort(key=lambda x: x[1], reverse=True)

        selected = scored[:top_k]

        if len(scored) > top_k:
            selected += random.sample(scored[top_k:], min(random_k, len(scored)-top_k))

        for r, _ in selected:
            new_data.append((prompt, r))

    return new_data

In [ ]:
def sft_update(data, steps=50):

    opt = torch.optim.Adam(policy.parameters(), lr=5e-6)

    for _ in range(steps):
        p, r = random.choice(data)

        text = p + " " + r
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

        outputs = policy.model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss

        opt.zero_grad()
        loss.backward()
        opt.step()

In [ ]:
for iteration in range(3):

    print(f"\n=== ITERATION {iteration} ===")

    # PPO phase
    for _ in range(2):
        for prompt in prompts:
            gen_ids, actions, logprobs, values, masks = collect_trajectory(prompt)

            text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            reward = compute_reward(prompt, text)

            rewards = [reward for _ in actions]
            returns = compute_gae(rewards, values, masks)

            buffer.states.append(gen_ids)
            buffer.actions.append(torch.stack(actions))
            buffer.logprobs.append(torch.stack(logprobs))
            buffer.values.append(torch.stack(values))
            buffer.returns.append(torch.stack(returns))

        ppo_update(buffer)
        buffer.clear()

    # self-improve
    new_data = self_improve(prompts)

    # SFT refine
    sft_update(new_data, steps=100)

    print("Iteration complete")